# World Cup 2026 — Saudi Arabia vs Uruguay
### AI-Powered Match Prediction
---

## Step 1 — Import Libraries

In [ ]:
import json
import os
import sys
sys.path.append('../src')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from dotenv import load_dotenv

load_dotenv('../.env')
print('✅ Libraries loaded successfully')

## Step 2 — Fetch Data from API-Football

In [ ]:
import requests

API_KEY = os.getenv('API_FOOTBALL_KEY')
HEADERS = {'x-apisports-key': API_KEY}
BASE    = 'https://v3.football.api-sports.io'

print('📡 Fetching data...')

sa  = requests.get(f'{BASE}/fixtures?team=523&last=10', headers=HEADERS).json()
ur  = requests.get(f'{BASE}/fixtures?team=26&last=10',  headers=HEADERS).json()
h2h = requests.get(f'{BASE}/fixtures/headtohead?h2h=523-26&last=10', headers=HEADERS).json()

print(f'✅ Saudi Arabia matches  : {len(sa["response"])}')
print(f'✅ Uruguay matches       : {len(ur["response"])}')
print(f'✅ Head to Head matches  : {len(h2h["response"])}')

## Step 3 — Process & Summarize Data

In [ ]:
def extract_stats(data, team_id):
    rows = []
    wins = draws = losses = goals_for = goals_against = 0
    for m in data.get('response', []):
        home    = m['teams']['home']
        away    = m['teams']['away']
        gh      = m['goals']['home'] or 0
        ga      = m['goals']['away'] or 0
        rows.append(f"{home['name']} {gh} - {ga} {away['name']}")
        is_home = home['id'] == team_id
        my_g    = gh if is_home else ga
        op_g    = ga if is_home else gh
        won     = home['winner'] if is_home else away['winner']
        goals_for     += my_g
        goals_against += op_g
        if won is True:    wins   += 1
        elif won is False: losses += 1
        else:              draws  += 1
    total   = wins + draws + losses or 1
    summary = {
        'wins': wins, 'draws': draws, 'losses': losses,
        'goals_for': goals_for, 'goals_against': goals_against,
        'avg_goals': round(goals_for / total, 1)
    }
    return rows, summary

sa_rows,  sa_stats  = extract_stats(sa,  523)
ur_rows,  ur_stats  = extract_stats(ur,  26)
h2h_rows, h2h_stats = extract_stats(h2h, 523)

print('🇸🇦 Saudi Arabia — Last 10:')
for r in sa_rows: print(f'   {r}')
print(f'   Summary: {sa_stats}\n')

print('🇺🇾 Uruguay — Last 10:')
for r in ur_rows: print(f'   {r}')
print(f'   Summary: {ur_stats}\n')

print('🤝 Head to Head:')
for r in h2h_rows: print(f'   {r}')

## Step 4 — Visualize Team Stats

In [ ]:
C_SA   = '#00A86B'
C_UR   = '#5EB2FF'
C_DRAW = '#6B7280'
C_GOLD = '#C8A951'
C_TEXT = '#E8EDF5'
C_BG   = '#0A0F1A'
C_MID  = '#161D2E'

fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor=C_BG)

# شارت 1: آخر 10 مباريات
ax = axes[0]
ax.set_facecolor(C_MID)
teams  = ['Saudi Arabia', 'Uruguay']
wins   = [sa_stats['wins'],   ur_stats['wins']]
draws  = [sa_stats['draws'],  ur_stats['draws']]
losses = [sa_stats['losses'], ur_stats['losses']]
x = range(len(teams))
b1 = ax.bar(x, wins,   color=C_SA,    label='Win',  width=0.5)
b2 = ax.bar(x, draws,  bottom=wins,   color=C_DRAW, label='Draw', width=0.5)
b3 = ax.bar(x, losses, bottom=[w+d for w,d in zip(wins,draws)],
            color='#DC2626', label='Loss', width=0.5)
ax.set_xticks(list(x))
ax.set_xticklabels(teams, color=C_TEXT)
ax.tick_params(colors=C_TEXT)
ax.spines[:].set_color('#2D3748')
ax.set_title('Last 10 Matches', color=C_GOLD, fontsize=12)
ax.legend(facecolor=C_MID, labelcolor=C_TEXT)
for bar in [b1, b2, b3]:
    for rect in bar:
        h = rect.get_height()
        if h > 0:
            ax.text(rect.get_x()+rect.get_width()/2, rect.get_y()+h/2,
                    str(int(h)), ha='center', va='center', color='white', fontweight='bold')

# شارت 2: مقارنة الأهداف
ax2 = axes[1]
ax2.set_facecolor(C_MID)
cats = ['Goals Scored', 'Goals Conceded']
sa_g = [sa_stats['goals_for'], sa_stats['goals_against']]
ur_g = [ur_stats['goals_for'], ur_stats['goals_against']]
x2   = range(len(cats))
w    = 0.35
ax2.bar([i-w/2 for i in x2], sa_g, width=w, color=C_SA, label='Saudi Arabia')
ax2.bar([i+w/2 for i in x2], ur_g, width=w, color=C_UR, label='Uruguay')
ax2.set_xticks(list(x2))
ax2.set_xticklabels(cats, color=C_TEXT)
ax2.tick_params(colors=C_TEXT)
ax2.spines[:].set_color('#2D3748')
ax2.set_title('Goals Comparison', color=C_GOLD, fontsize=12)
ax2.legend(facecolor=C_MID, labelcolor=C_TEXT)
for bars in ax2.containers:
    ax2.bar_label(bars, color=C_TEXT, padding=2)

fig.suptitle('Team Statistics — Last 10 Matches', color=C_GOLD, fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/stats_chart.png', dpi=150, bbox_inches='tight', facecolor=C_BG)
plt.show()
print('✅ Stats chart saved')

## Step 5 — Claude AI Prediction

In [ ]:
import anthropic

ANTHROPIC_KEY = os.getenv('ANTHROPIC_KEY')

prompt = f"""You are an expert football analyst. Analyze Saudi Arabia vs Uruguay in World Cup 2026.

=== Saudi Arabia Last 10 ===
{chr(10).join(sa_rows)}
Summary: W{sa_stats['wins']} D{sa_stats['draws']} L{sa_stats['losses']} | GF:{sa_stats['goals_for']} GA:{sa_stats['goals_against']}

=== Uruguay Last 10 ===
{chr(10).join(ur_rows)}
Summary: W{ur_stats['wins']} D{ur_stats['draws']} L{ur_stats['losses']} | GF:{ur_stats['goals_for']} GA:{ur_stats['goals_against']}

=== Head to Head ===
{chr(10).join(h2h_rows)}

=== Extra Info ===
- Uruguay arrived 24h late due to flight issues
- Match in Miami (hot weather)
- Saudi Arabia beat Argentina in 2022

Reply ONLY with JSON, no extra text:
{{
  "score_sa": number,
  "score_ur": number,
  "prob_sa": number 0-100,
  "prob_draw": number 0-100,
  "prob_ur": number 0-100,
  "winner": "sa" or "draw" or "ur",
  "analysis": "3 sentences in English",
  "key_factor": "most decisive factor"
}}
Make sure prob_sa + prob_draw + prob_ur = 100"""

print('🤖 Claude is analyzing...')
client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
resp   = client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=1000,
    messages=[{'role': 'user', 'content': prompt}]
)
raw   = resp.content[0].text
clean = raw.replace('```json','').replace('```','').strip()
pred  = json.loads(clean)

os.makedirs('../outputs', exist_ok=True)
with open('../outputs/prediction.json', 'w', encoding='utf-8') as f:
    json.dump(pred, f, ensure_ascii=False, indent=2)

print('✅ Prediction saved to outputs/prediction.json\n')
print(f"Score     : KSA {pred['score_sa']} - {pred['score_ur']} URU")
print(f"KSA Win   : {pred['prob_sa']}%")
print(f"Draw      : {pred['prob_draw']}%")
print(f"URU Win   : {pred['prob_ur']}%")
print(f"Analysis  : {pred['analysis']}")
print(f"Key Factor: {pred['key_factor']}")

## Step 6 — Visualize Prediction

In [ ]:
fig = plt.figure(figsize=(14, 10), facecolor=C_BG)
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# شارت 1: Donut
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(C_MID)
sizes = [pred['prob_sa'], pred['prob_draw'], pred['prob_ur']]
ax1.pie(sizes, colors=[C_SA, C_DRAW, C_UR], startangle=90,
        wedgeprops=dict(width=0.55, edgecolor=C_BG, linewidth=2))
ax1.text(0, 0.12, f"{pred['prob_sa']}%", ha='center', fontsize=15,
         fontweight='bold', color=C_SA)
ax1.text(0, -0.15, 'KSA', ha='center', fontsize=9, color=C_TEXT)
patches = [
    mpatches.Patch(color=C_SA,   label=f"KSA {pred['prob_sa']}%"),
    mpatches.Patch(color=C_DRAW, label=f"Draw {pred['prob_draw']}%"),
    mpatches.Patch(color=C_UR,   label=f"URU {pred['prob_ur']}%"),
]
ax1.legend(handles=patches, loc='lower center', bbox_to_anchor=(0.5,-0.28),
           fontsize=8, facecolor=C_MID, labelcolor=C_TEXT)
ax1.set_title('Win Probability', color=C_GOLD, fontsize=11, pad=10)

# شارت 2: النتيجة
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(C_MID)
ax2.axis('off')
ax2.text(0.5, 0.85, 'Predicted Score', ha='center', fontsize=11,
         color=C_GOLD, transform=ax2.transAxes)
ax2.text(0.5, 0.52, f"{pred['score_sa']}  -  {pred['score_ur']}",
         ha='center', fontsize=38, fontweight='bold',
         color=C_TEXT, transform=ax2.transAxes)
ax2.text(0.25, 0.28, '🇸🇦 KSA', ha='center', fontsize=11,
         color=C_SA, transform=ax2.transAxes)
ax2.text(0.75, 0.28, 'URU 🇺🇾', ha='center', fontsize=11,
         color=C_UR, transform=ax2.transAxes)
winner_map = {'sa': (f"KSA Wins 🇸🇦", C_SA),
              'draw': ('Draw 🤝', C_DRAW),
              'ur': ('URU Wins 🇺🇾', C_UR)}
wlabel, wcolor = winner_map[pred['winner']]
ax2.text(0.5, 0.10, wlabel, ha='center', fontsize=10,
         color=wcolor, fontweight='bold', transform=ax2.transAxes)
ax2.set_title('Final Prediction', color=C_GOLD, fontsize=11, pad=10)

# شارت 3: آخر 10
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor(C_MID)
x3 = range(2)
b1 = ax3.bar(x3, [sa_stats['wins'],   ur_stats['wins']],   color=C_SA,    label='Win',  width=0.5)
b2 = ax3.bar(x3, [sa_stats['draws'],  ur_stats['draws']],
             bottom=[sa_stats['wins'], ur_stats['wins']],   color=C_DRAW,  label='Draw', width=0.5)
b3 = ax3.bar(x3, [sa_stats['losses'], ur_stats['losses']],
             bottom=[sa_stats['wins']+sa_stats['draws'], ur_stats['wins']+ur_stats['draws']],
             color='#DC2626', label='Loss', width=0.5)
ax3.set_xticks(list(x3))
ax3.set_xticklabels(['Saudi Arabia', 'Uruguay'], color=C_TEXT, fontsize=9)
ax3.tick_params(colors=C_TEXT)
ax3.spines[:].set_color('#2D3748')
ax3.set_title('Last 10 Matches', color=C_GOLD, fontsize=11, pad=10)
ax3.legend(fontsize=8, facecolor=C_MID, labelcolor=C_TEXT)
for bar in [b1, b2, b3]:
    for rect in bar:
        h = rect.get_height()
        if h > 0:
            ax3.text(rect.get_x()+rect.get_width()/2, rect.get_y()+h/2,
                     str(int(h)), ha='center', va='center', color='white', fontweight='bold')

# شارت 4: الأهداف
ax4 = fig.add_subplot(gs[1, 0])
ax4.set_facecolor(C_MID)
cats = ['Goals Scored', 'Goals Conceded']
x4   = range(2)
w    = 0.35
ax4.bar([i-w/2 for i in x4], [sa_stats['goals_for'], sa_stats['goals_against']],
        width=w, color=C_SA, label='Saudi Arabia')
ax4.bar([i+w/2 for i in x4], [ur_stats['goals_for'], ur_stats['goals_against']],
        width=w, color=C_UR, label='Uruguay')
ax4.set_xticks(list(x4))
ax4.set_xticklabels(cats, color=C_TEXT, fontsize=9)
ax4.tick_params(colors=C_TEXT)
ax4.spines[:].set_color('#2D3748')
ax4.set_title('Goals Stats', color=C_GOLD, fontsize=11, pad=10)
ax4.legend(fontsize=8, facecolor=C_MID, labelcolor=C_TEXT)
for bars in ax4.containers:
    ax4.bar_label(bars, color=C_TEXT, fontsize=9, padding=2)

# شارت 5: التحليل
ax5 = fig.add_subplot(gs[1, 1:])
ax5.set_facecolor(C_MID)
ax5.axis('off')
ax5.text(0.5, 0.95, 'AI Analysis', ha='center', fontsize=11,
         color=C_GOLD, transform=ax5.transAxes, fontweight='bold')
ax5.text(0.5, 0.70, pred['analysis'], ha='center', va='top',
         fontsize=9.5, color=C_TEXT, transform=ax5.transAxes,
         multialignment='center',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#1E293B', edgecolor='#2D3748'))
ax5.text(0.5, 0.25, f"Key Factor: {pred['key_factor']}",
         ha='center', fontsize=9.5, color=C_GOLD, transform=ax5.transAxes,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#1A1A00', edgecolor=C_GOLD, alpha=0.8))

fig.text(0.5, 0.97, '⚽  FIFA World Cup 2026  |  🇸🇦 Saudi Arabia vs Uruguay 🇺🇾',
         ha='center', fontsize=14, fontweight='bold', color=C_GOLD)
fig.text(0.5, 0.935, 'AI-Powered Prediction — Real Data from API-Football',
         ha='center', fontsize=9, color='#7A8BAA')

plt.savefig('../outputs/prediction_chart.png', dpi=150, bbox_inches='tight', facecolor=C_BG)
plt.show()
print('✅ Prediction chart saved to outputs/prediction_chart.png')